# PART 1: SETUP & CONFIGURATION

In [2]:
import os
import glob
import json
import numpy as np
from pathlib import Path
from typing import List, Tuple
import cv2
from PIL import Image
import shutil

# ============================================================================
# PART 1: SETUP & CONFIGURATION
# ============================================================================

print("=" * 80)
print("PART 1: Setup & Configuration")
print("=" * 80)

# ✏️ UPDATE THESE FOR YOUR SETUP:
IMAGE_FOLDER = "/kaggle/input/datasets/appoline/appoline"  # ← Change this to your dataset path
OUTPUT_FOLDER = "/kaggle/working/lora_output/"      # Where results save
TRAINING_FOLDER = "/kaggle/working/training_data/" # Temp folder for preprocessing

# Create output directories
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(TRAINING_FOLDER, exist_ok=True)

# Configuration for training
CONFIG = {
    "character_name": "appoline",  # ← Change this to your character name
    "trigger_word": "apl",         # ← The activation token (2-3 words, unique)
    "base_model": "runwayml/stable-diffusion-v1-5",  # SD 1.5 (good balance)
    # Alternatives if you want to try:
    #   - "stabilityai/stable-diffusion-2-1" (SD 2.1, better for faces)
    #   - "stabilityai/stable-diffusion-xl-base-1.0" (SDXL, higher quality, more VRAM)
    
    # Training hyperparameters (these control *how* the model learns)
    "num_train_epochs": 100,       # How many times it sees all images
    "train_batch_size": 4,          # Images per step (↓ if GPU runs out of memory)
    "learning_rate": 1e-4,          # How fast it learns (too high = bad, too low = slow)
    "rank": 16,                     # LoRA size (16 = small/fast, 32 = larger/detailed)
    "target_modules": ["to_k", "to_v", "to_q", "to_out.0"],  # Which layers to adapt
    "image_size": 512,              # Resolution (512x512 is standard, 768+ needs more VRAM)
}

print(f"\n✅ Configuration:")
print(f"   Character: {CONFIG['character_name']}")
print(f"   Trigger word: '{CONFIG['trigger_word']}'")
print(f"   Base model: {CONFIG['base_model']}")
print(f"   Image folder: {IMAGE_FOLDER}")
print(f"\n⚠️  Remember: You'll use the trigger word when generating images:")
print(f"   'A photo of {CONFIG['trigger_word']}, detailed face, studio lighting'")

PART 1: Setup & Configuration

✅ Configuration:
   Character: appoline
   Trigger word: 'apl'
   Base model: runwayml/stable-diffusion-v1-5
   Image folder: /kaggle/input/datasets/appoline/appoline

⚠️  Remember: You'll use the trigger word when generating images:
   'A photo of apl, detailed face, studio lighting'


# PART 2: DATA PREPARATION

In [3]:
print("\n" + "=" * 80)
print("PART 2: Data Preparation")
print("=" * 80)

def validate_images(folder: str) -> List[str]:
    """
    Find all valid image files in folder.
    
    Why this matters:
    - Some files might be corrupted or not actual images
    - We need to know how many valid images we have
    - Corrupted images crash training
    """
    valid_formats = ("*.jpg", "*.jpeg", "*.png", "*.webp")
    images = []
    
    for fmt in valid_formats:
        images.extend(glob.glob(os.path.join(folder, "**", fmt), recursive=True))
    
    print(f"\n📁 Found {len(images)} images in {folder}")
    
    # Check each image is actually valid
    valid_images = []
    for img_path in images:
        try:
            img = Image.open(img_path)
            img.verify()  # This will raise if corrupted
            valid_images.append(img_path)
        except Exception as e:
            print(f"   ⚠️  Skipped {os.path.basename(img_path)}: {e}")
    
    print(f"✅ {len(valid_images)} valid images ready for training")
    
    if len(valid_images) < 10:
        print(f"\n⚠️  WARNING: You have only {len(valid_images)} images.")
        print("    For best results, aim for 50-100 diverse images of your character.")
        print("    You can generate more using the base model first, then train!")
    
    return valid_images

# Find your images
images = validate_images(IMAGE_FOLDER)

if not images:
    raise ValueError(f"❌ No images found in {IMAGE_FOLDER}. Check the path!")


PART 2: Data Preparation

📁 Found 15 images in /kaggle/input/datasets/appoline/appoline
✅ 15 valid images ready for training


# PART 3: OPTIONAL - FACE CROPPING (for better training)

In [ ]:
print("\n" + "=" * 80)
print("PART 3: Face Detection & Cropping (Optional but Recommended)")
print("=" * 80)

try:
    from insightface.app import FaceAnalysis
    
    print("\n⏳ Installing face detection model...")
    face_detector = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
    face_detector.prepare(ctx_id=-1, det_size=(640, 640))
    
    def crop_face_from_image(img_path: str, output_path: str, padding: float = 0.2) -> bool:
        """
        Crop the face region from an image for better LoRA training.
        
        Why cropping helps:
        - LoRA learns faster when focused on the face (not background)
        - Cleaner training = more consistent character
        - Reduces file size
        
        Args:
            img_path: Path to original image
            output_path: Where to save cropped image
            padding: How much space around face (0.2 = 20% padding)
        
        Returns:
            True if successful, False if no face found
        """
        try:
            img = cv2.imread(img_path)
            if img is None:
                return False
            
            # Detect faces
            faces = face_detector.get(img)
            if len(faces) == 0:
                return False
            
            # Get the largest/most prominent face
            face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
            x1, y1, x2, y2 = [int(v) for v in face.bbox]
            
            # Add padding around face
            h, w = img.shape[:2]
            pad_x = int((x2 - x1) * padding)
            pad_y = int((y2 - y1) * padding)
            
            x1 = max(0, x1 - pad_x)
            y1 = max(0, y1 - pad_y)
            x2 = min(w, x2 + pad_x)
            y2 = min(h, y2 + pad_y)
            
            # Crop and save
            cropped = img[y1:y2, x1:x2]
            cv2.imwrite(output_path, cropped)
            return True
        except:
            return False
    
    # Process all images
    print(f"\n⏳ Cropping faces from {len(images)} images...")
    cropped_count = 0
    
    for i, img_path in enumerate(images):
        filename = os.path.basename(img_path)
        output_path = os.path.join(TRAINING_FOLDER, filename)
        
        if crop_face_from_image(img_path, output_path):
            cropped_count += 1
        else:
            # If cropping fails, use original (no face detected, that's okay)
            shutil.copy(img_path, output_path)
        
        if (i + 1) % max(1, len(images) // 5) == 0:
            print(f"   Progress: {i + 1}/{len(images)}")
    
    print(f"✅ Cropped {cropped_count}/{len(images)} faces successfully")
    
except ImportError:
    print("\n⚠️  insightface not available, using original images")
    print("   (This is fine! Just copy images to training folder)")
    for img_path in images:
        shutil.copy(img_path, TRAINING_FOLDER)

# Verify training data
training_images = glob.glob(os.path.join(TRAINING_FOLDER, "*.*"))
print(f"\n✅ Training data ready: {len(training_images)} images")

# PART 4: INSTALL DEPENDENCIES

In [4]:
print("\n" + "=" * 80)
print("PART 4: Installing Dependencies")
print("=" * 80)

print("\n⏳ Installing required libraries...")
print("   (This takes ~2-3 minutes, run once per session)")

import subprocess
import sys

dependencies = [
    "diffusers==0.21.4",          # The main diffusion library
    "transformers==4.30.2",        # For the CLIP text encoder
    "safetensors==0.3.1",          # Safe model format
    "accelerate==0.20.3",          # GPU acceleration
    "peft==0.4.0",                 # LoRA implementation (Parameter-Efficient Fine-Tuning)
    "pillow==9.5.0",               # Image processing
    "opencv-python==4.7.0.72",     # Face detection if available
]

for package in dependencies:
    print(f"   Installing {package.split('==')[0]}...")
    subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", 
    "--only-binary=:all:",  # ← Use pre-compiled binary, never build
    "tokenizers"
])

print("✅ All dependencies installed!")


PART 4: Installing Dependencies

⏳ Installing required libraries...
   (This takes ~2-3 minutes, run once per session)
   Installing diffusers...
   Installing transformers...
   Installing safetensors...
   Installing accelerate...
   Installing peft...
   Installing pillow...
   Installing opencv-python...
✅ All dependencies installed!


# PART 5: LoRA TRAINING

In [5]:
print("\n" + "=" * 80)
print("PART 5: LoRA Training")
print("=" * 80)

print(f"""
🎓 What happens during training:
   1. Load the base model ({CONFIG['base_model']})
   2. For each image, generate a description (e.g., "a photo of {CONFIG['trigger_word']}")
   3. Calculate how the model's output differs from the training image
   4. Update the LoRA weights to match your character better
   5. Repeat {CONFIG['num_train_epochs']} times across all images

This teaches the model: "When I see {CONFIG['trigger_word']}, generate THIS character"

Estimated time: 15 mins (15 images) to 2 hours (100 images)
GPU needed: Yes (Kaggle P100 GPU)
""")

import torch
from diffusers import StableDiffusionPipeline
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# Check GPU
print(f"\n🖥️  GPU Check:")
print(f"   PyTorch sees GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  No GPU detected! Training will be VERY slow.")

# Create a simple dataset for training
class CharacterImageDataset(Dataset):
    """
    Custom dataset for loading character images.
    
    Why a custom dataset?
    - PyTorch training loops need data in a specific format
    - This handles loading, resizing, and preparing images
    - Allows batching (processing multiple images at once for speed)
    """
    
    def __init__(self, image_folder: str, trigger_word: str, size: int = 512):
        self.image_paths = glob.glob(os.path.join(image_folder, "*.*"))
        self.trigger_word = trigger_word
        self.size = size
        
        # Transform: resize to 512x512, convert to tensor, normalize
        self.transforms = transforms.Compose([
            transforms.Resize((size, size), interpolation=Image.LANCZOS),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # Normalize to [-1, 1]
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img_tensor = self.transforms(img)
        
        # Create a text description for this image
        # The model will learn to generate this from the image
        prompt = f"a photo of {self.trigger_word}"
        
        return {
            "pixel_values": img_tensor,
            "input_ids": prompt,  # Will be tokenized by the pipeline
        }

# Initialize dataset and dataloader
print(f"\n📚 Creating dataset...")
dataset = CharacterImageDataset(
    TRAINING_FOLDER,
    CONFIG["trigger_word"],
    CONFIG["image_size"]
)

dataloader = DataLoader(
    dataset,
    batch_size=CONFIG["train_batch_size"],
    shuffle=True
)

print(f"   Dataset size: {len(dataset)} images")
print(f"   Batch size: {CONFIG['train_batch_size']}")
print(f"   Batches per epoch: {len(dataloader)}")

# Load the base model
print(f"\n🤖 Loading base model: {CONFIG['base_model']}")
print("   (First time downloads ~5GB, then cached)")

device = "cuda" if torch.cuda.is_available() else "cpu"

pipeline = StableDiffusionPipeline.from_pretrained(
    CONFIG["base_model"],
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,  # Disable for faster training
)
pipeline = pipeline.to(device)

# Enable memory optimizations
if device == "cuda":
    pipeline.enable_attention_slicing()  # Reduces VRAM usage
    # pipeline.enable_xformers_memory_efficient_attention()  # Uncomment if available

print("✅ Model loaded!")

# Create LoRA configuration
print(f"\n⚙️  Configuring LoRA...")
print(f"   Rank: {CONFIG['rank']} (16=small/fast, 32=detailed)")
print(f"   Target modules: {CONFIG['target_modules']}")

lora_config = LoraConfig(
    r=CONFIG["rank"],
    lora_alpha=CONFIG["rank"] * 2,  # Scaling factor
    target_modules=CONFIG["target_modules"],
    lora_dropout=0.1,
    bias="none",
    task_type="image_generation",
)

# Wrap model with LoRA
unet = pipeline.unet
unet = get_peft_model(unet, lora_config)

# Count trainable parameters
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in unet.parameters())

print(f"\n📊 Model Size:")
print(f"   Total parameters: {total_params / 1e6:.1f}M")
print(f"   Trainable (LoRA): {trainable_params / 1e6:.1f}M ({trainable_params/total_params*100:.2f}%)")
print(f"   → This is why LoRA is efficient! Only 0.1-1% of weights are trained.")

# Set up optimizer
from torch.optim import AdamW

optimizer = AdamW(
    unet.parameters(),
    lr=CONFIG["learning_rate"]
)

print(f"\n🎯 Training Configuration:")
print(f"   Learning rate: {CONFIG['learning_rate']}")
print(f"   Epochs: {CONFIG['num_train_epochs']}")
print(f"   Total steps: {len(dataloader) * CONFIG['num_train_epochs']}")

print(f"\n⏳ Starting training...")
print(f"   This will take 15 minutes - 2 hours depending on your GPU and image count")

# Training loop
loss_history = []
best_loss = float('inf')

for epoch in range(CONFIG["num_train_epochs"]):
    unet.train()
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['num_train_epochs']}")
    
    for batch in progress_bar:
        # Move batch to GPU
        pixel_values = batch["pixel_values"].to(device)
        
        # Encode prompts to tokens (dummy for now, real implementation would tokenize)
        # In production, you'd use the tokenizer from the model
        
        # Forward pass through the diffusion model
        # This is simplified - real training would do the full diffusion loss
        try:
            with torch.no_grad():
                latents = pipeline.vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215
            
            # Get random timesteps for diffusion
            timesteps = torch.randint(
                0, pipeline.scheduler.config.num_train_timesteps,
                (pixel_values.shape[0],),
                device=device
            )
            
            # Add noise (forward diffusion process)
            noise = torch.randn_like(latents)
            noisy_latents = pipeline.scheduler.add_noise(latents, noise, timesteps)
            
            # Predict noise with UNet (the part we're training)
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=None,  # Simplified
            ).sample
            
            # Loss: how far off our prediction is
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            
            # Backward pass: calculate gradients
            loss.backward()
            
            # Update weights
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += loss.detach().item()
            progress_bar.set_postfix({"loss": total_loss / (progress_bar.n + 1)})
            
        except Exception as e:
            print(f"\n⚠️  Training step error: {e}")
            print("   This might happen if GPU runs out of memory.")
            print("   Try reducing train_batch_size or image_size")
            break
    
    # Save checkpoint
    avg_loss = total_loss / len(dataloader)
    loss_history.append(avg_loss)
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"✅ Epoch {epoch+1}: Loss = {avg_loss:.4f} (improved!)")
        
        # Save LoRA weights
        unet.save_pretrained(os.path.join(OUTPUT_FOLDER, "lora_weights"))
    else:
        print(f"   Epoch {epoch+1}: Loss = {avg_loss:.4f}")

print("\n✅ Training complete!")


PART 5: LoRA Training

🎓 What happens during training:
   1. Load the base model (runwayml/stable-diffusion-v1-5)
   2. For each image, generate a description (e.g., "a photo of apl")
   3. Calculate how the model's output differs from the training image
   4. Update the LoRA weights to match your character better
   5. Repeat 100 times across all images

This teaches the model: "When I see apl, generate THIS character"

Estimated time: 15 mins (15 images) to 2 hours (100 images)
GPU needed: Yes (Kaggle P100 GPU)



/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()
Flax classes are deprecated and will be 


🖥️  GPU Check:
   PyTorch sees GPU: True
   GPU: Tesla P100-PCIE-16GB
   VRAM: 17.1 GB

📚 Creating dataset...


ValueError: num_samples should be a positive integer value, but got num_samples=0

# PART 6: SAVE THE LoRA

In [ ]:
print("\n" + "=" * 80)
print("PART 6: Save & Export LoRA")
print("=" * 80)

# Save as safetensors (the standard format)
from safetensors.torch import save_file

lora_weights = unet.state_dict()
lora_path = os.path.join(OUTPUT_FOLDER, f"{CONFIG['character_name']}-lora.safetensors")

print(f"\n💾 Saving LoRA as safetensors...")
save_file(lora_weights, lora_path)

# Add metadata so other tools know what trigger word to use
metadata = {
    "ss_trained_words": CONFIG["trigger_word"],
    "ss_sd_model_name": CONFIG["base_model"],
    "ss_network_dim": str(CONFIG["rank"]),
    "ss_max_train_resolution": f"{CONFIG['image_size']}x{CONFIG['image_size']}",
}

print(f"\n📦 LoRA saved to: {lora_path}")
print(f"\n✅ READY TO USE!")
print(f"\nWhen generating images, use the prompt:")
print(f'   "a photo of {CONFIG["trigger_word"]}, detailed face, professional lighting"')

# PART 7: TEST THE LoRA

In [ ]:
print("\n" + "=" * 80)
print("PART 7: Generate Test Images")
print("=" * 80)

print(f"""
🎨 Testing your LoRA:
   
   Let's generate a few test images to see if your LoRA works.
   If the character looks consistent, training was successful!
   If not, you might need:
   - More training images (100+ is ideal)
   - Different learning rate
   - More training epochs
""")

# Load the trained LoRA back into the pipeline
from peft import PeftModel

print(f"\n⏳ Loading trained LoRA...")
pipeline.unet = PeftModel.from_pretrained(
    pipeline.unet,
    os.path.join(OUTPUT_FOLDER, "lora_weights")
)

# Generate test images
test_prompts = [
    f"a portrait of {CONFIG['trigger_word']}, detailed face, studio lighting",
    f"{CONFIG['trigger_word']}, full body, outdoor, professional photography",
    f"a close-up of {CONFIG['trigger_word']}, realistic skin, natural lighting",
]

print(f"\n🎨 Generating {len(test_prompts)} test images...")

for i, prompt in enumerate(test_prompts):
    print(f"\n   [{i+1}/{len(test_prompts)}] {prompt}")
    
    image = pipeline(
        prompt,
        height=CONFIG["image_size"],
        width=CONFIG["image_size"],
        num_inference_steps=50,
        guidance_scale=7.5,
    ).images[0]
    
    # Save test image
    test_image_path = os.path.join(OUTPUT_FOLDER, f"test_{i+1}.png")
    image.save(test_image_path)
    print(f"   ✅ Saved to {test_image_path}")

print(f"\n✅ All done! Check the test images in {OUTPUT_FOLDER}")
print(f"\nNext steps:")
print(f"1. Review test images — do they look like your character?")
print(f"2. Download the LoRA file: {lora_path}")
print(f"3. Use in ComfyUI, Automatic1111, or other SD frontends")
print(f"4. If quality isn't great, collect more images and retrain!")